In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption\\research'

In [3]:
# Changing working directory one path back

os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    image_feex_path: Path           
    main_model_path: Path   

    IMAGE_SIZE: list         
    WEIGHTS: str            
    INCLUDE_TOP: bool        
    POOLING: str

    CNN_DIM: int
    VOCAB_SIZE: int          
    MAX_LENGTH: int          
    EMBEDDING_DIM: int       
    LSTM_UNITS: int              
    DROPOUT: float          


In [6]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        params = self.params

        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
                                                root_dir=Path(config.root_dir),
                                                image_feex_path=Path(config.image_feex_path),
                                                main_model_path=Path(config.main_model_path),
                                                IMAGE_SIZE=params.IMAGE_SIZE,
                                                WEIGHTS=params.WEIGHTS,
                                                INCLUDE_TOP=params.INCLUDE_TOP,
                                                POOLING=params.POOLING,
                                                CNN_DIM=params.CNN_DIM,
                                                VOCAB_SIZE=params.VOCAB_SIZE,
                                                MAX_LENGTH=params.MAX_LENGTH,
                                                EMBEDDING_DIM=params.EMBEDDING_DIM,
                                                LSTM_UNITS=params.LSTM_UNITS,
                                                DROPOUT=params.DROPOUT
                                            )

        return prepare_base_model_config

In [8]:
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout,Concatenate                           
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from imgCaption import logger

In [9]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def get_feature_ex_model(self):
        self.model = tf.keras.applications.resnet50.ResNet50(
                                input_shape=self.config.IMAGE_SIZE,  
                                weights=self.config.WEIGHTS,                 
                                include_top=self.config.INCLUDE_TOP,         
                                pooling=self.config.POOLING                  
                                                            )
        self.model.trainable = False
        self.save_model(path=self.config.image_feex_path, model=self.model)
        logger.info("Feature Extraction model saved")

    @staticmethod
    def prepare_full_model(cnn_dim, vocab_size,max_caption_length,embedding_dim,
                            lstm_units,dropout_rate):
        
        input_img_features = Input(shape=(cnn_dim,), name = "Img_Features_Input")
        img_drop_layer = Dropout(dropout_rate, name="img_drop_layer")(input_img_features)
        img_output_tensor = Dense(embedding_dim, activation='relu', name="img_output_tensor")(img_drop_layer)
        
        input_caption = Input(shape=(max_caption_length,), name="Caption_Input")
        cap_embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="cap_embedding_layer")(input_caption)
        cap_drop_layer = Dropout(dropout_rate, name="cap_drop_layer")(cap_embedding_layer)
        cap_lstm_1 = LSTM(lstm_units, return_sequences=True, name="cap_lstm_1")(cap_drop_layer)
        cap_lstm_1_drop = Dropout(dropout_rate, name="cap_lstm_1_drop")(cap_lstm_1)
        cap_output_tensor = LSTM(lstm_units, name="cap_output_tensor_LSTM")(cap_lstm_1_drop)
    
        merged_tensor = Concatenate(axis=-1)([img_output_tensor,cap_output_tensor])

        merged_layer = Dense(embedding_dim, activation='relu', name="merged_layer")(merged_tensor)
        merged_drop_layer = Dropout(dropout_rate, name="merged_drop_layer")(merged_layer)
        output_layer = Dense(vocab_size, activation='softmax', name="Output_Layer_Final")(merged_drop_layer)

        full_model = Model(
            inputs=[input_img_features, input_caption],
            outputs=output_layer,
            name='Image_Captioning'
        )

        full_model.summary()
        return full_model

    def main_model(self):
        self.full_model = self.prepare_full_model(
                            cnn_dim=self.config.CNN_DIM,                               
                            vocab_size=self.config.VOCAB_SIZE,
                            max_caption_length=self.config.MAX_LENGTH,
                            embedding_dim=self.config.EMBEDDING_DIM,
                            lstm_units=self.config.LSTM_UNITS,
                            dropout_rate=self.config.DROPOUT
                            )
        self.save_model(path=self.config.main_model_path, model=self.full_model)
        logger.info("Main model saved")

In [10]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_feature_ex_model()
    main_model=prepare_base_model.main_model()
except Exception as e:
    raise e

[2026-07-07 12:07:26,280: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-07 12:07:26,284: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-07 12:07:26,290: INFO: common: created directory at: artifacts]
[2026-07-07 12:07:26,293: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-07-07 12:07:28,420: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
[2026-07-07 12:07:28,973: INFO: 791305037: Feature Extraction model saved]


Model: "Image_Captioning"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Caption_Input       │ (None, 37)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_embedding_layer │ (None, 37, 256)   │  2,248,192 │ Caption_Input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_drop_layer      │ (None, 37, 256)   │          0 │ cap_embedding_la… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 37)        │          0 │ Caption_Input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Img_Features_Input  │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_lstm_1 (LSTM)   │ (None, 37, 512)   │  1,574,912 │ cap_drop_layer[0… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_drop_layer      │ (None, 2048)      │          0 │ Img_Features_Inp… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_lstm_1_drop     │ (None, 37, 512)   │          0 │ cap_lstm_1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_output_tensor   │ (None, 256)       │    524,544 │ img_drop_layer[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_output_tensor_… │ (None, 512)       │  2,099,200 │ cap_lstm_1_drop[… │
│ (LSTM)              │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 768)       │          0 │ img_output_tenso… │
│ (Concatenate)       │                   │            │ cap_output_tenso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_layer        │ (None, 256)       │    196,864 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_drop_layer   │ (None, 256)       │          0 │ merged_layer[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer_Final  │ (None, 8782)      │  2,256,974 │ merged_drop_laye… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,900,686 (33.95 MB)

 Trainable params: 8,900,686 (33.95 MB)

 Non-trainable params: 0 (0.00 B)

[2026-07-07 12:07:29,630: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
[2026-07-07 12:07:29,723: INFO: 791305037: Main model saved]
